In [ ]:
import arcpy
arcpy.env.workspace = r"C:\Users\leopo\OneDrive\IFGI\2026_SS\Python in QGIS and ArcGIS\exercise_arcpy_1.gdb"
arcpy.env.overwriteOutput = True

In [ ]:
point_fcs = arcpy.ListFeatureClasses(feature_type="Point")

fields = ["SHAPE@", "status", "type"]
with arcpy.da.InsertCursor("active_assets", fields) as i_cur:
    for fc in point_fcs:
        if fc == "active_assets":
            continue
        with arcpy.da.SearchCursor(fc, fields, where_clause="status = 'active'") as s_cur:
            for row in s_cur:
                i_cur.insertRow(row)
    i_cur.delete()

In [ ]:
import os
SIZES = {"mast": "300 Meters", "mobile_antenna": "50 Meters", "building_antenna": "100  Meters"}

buffers = []
for t, dist in SIZES.items():
    lyr = f"lyr_{t}"
    arcpy.management.MakeFeatureLayer("active_assets", lyr, f"type = '{t}'")
    out = os.path.join(arcpy.env.workspace, f"buf_{t}")
    arcpy.analysis.Buffer(lyr, out, dist)
    buffers.append(out)

arcpy.management.Merge(buffers, "coverage")